# LAB2


## INSTALACJA I IMPORT BIBLOTEK


In [1]:
import pandas as pd
import geopandas as gpd
import folium
from IPython.display import display

In [5]:
df = pd.read_csv("/content/data/pm25_sensors.csv")
districts = gpd.read_file("/content/data/opole_districts.geojson")
schools = gpd.read_file("/content/data/opole_schools.geojson")

display(df.head(10))
display(districts.head())
display(schools.head())

,sensor_id,lat,lon,pm25,measured_at
0,PM-042,50.675603,17.906425,23.5,2026-10-01 08:36:00
1,PM-106,50.632283,17.919674,12.4,2026-10-01 08:22:00
2,PM-130,50.635608,18.020378,19.4,2026-10-01 08:24:00
3,PM-006,50.674051,NaN,25.7,2026-10-01 08:03:00
4,PM-025,50.676438,17.938766,16.8,2026-10-01 08:13:00
5,PM-137,17.873510,50.677587,14.4,2026-10-01 08:06:00
6,PM-036,50.666829,17.940071,23.7,2026-10-01 08:23:00
7,PM-100,50.694328,17.802222,12.5,2026-10-01 08:56:00
8,PM-092,50.684393,17.794548,9.3,2026-10-01 08:09:00
9,PM-087,50.650288,17.940259,-999.0,2026-10-01 08:56:00


,osm_id,name,geometry
0,9709955,Armii Krajowej,"POLYGON ((17.94533 50.68232, 17.94712 50.68404..."
1,9710023,Bierkowice,"POLYGON ((17.84376 50.6998, 17.84415 50.69965,..."
2,9710193,Borki,"POLYGON ((17.87562 50.74302, 17.87767 50.74222..."
3,9710322,Brzezie,"POLYGON ((17.88652 50.75983, 17.88756 50.75843..."
4,9711126,Chabry,"POLYGON ((17.92146 50.67526, 17.92164 50.67542..."


,osm_id,name,geometry
0,way/207027259,Niepubliczna Szkoła Podstawowa Nasza Szkoła,POINT (17.93454 50.68191)
1,way/208494323,Publiczna Szkoła Podstawowa nr 1 im. Obrońców ...,POINT (17.93088 50.66443)
2,way/452471787,Publiczna Szkoła Podstawowa nr 10 im. Henryka ...,POINT (17.90108 50.65643)
3,way/1102360496,Publiczna Szkoła Podstawowa nr 11 im. Orląt Lw...,POINT (17.92788 50.68001)
4,way/207020514,Publiczna Szkoła Podstawowa nr 15,POINT (17.95549 50.67215)


## ANALIZA DANYCH

In [6]:
print("Liczba rekordów:", len(df))
print("\nTypy danych:")
df.info()

print("\nStatystyki:")
display(df.describe())

print("\nBrakujące wartości:")
print(df.isna().sum())

print("\nDuplikaty:", df.duplicated().sum())
print("\nBrak wartości")
display(df[df.isna().any(axis=1)])

Liczba rekordów: 145

Typy danych:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sensor_id    145 non-null    object 
 1   lat          142 non-null    float64
 2   lon          141 non-null    float64
 3   pm25         145 non-null    float64
 4   measured_at  145 non-null    object 
dtypes: float64(3), object(2)
memory usage: 5.8+ KB

Statystyki:


,lat,lon,pm25
count,142.000000,141.000000,145.000000
mean,49.980869,18.613736,-8.995172
std,4.727557,4.743171,167.401474
min,17.873510,17.785622,-999.000000
25%,50.652463,17.897319,14.500000
50%,50.669682,17.918011,19.100000
75%,50.686753,17.940870,22.500000
max,50.767895,50.677587,35.300000



Brakujące wartości:
sensor_id      0
lat            3
lon            4
pm25           0
measured_at    0
dtype: int64

Duplikaty: 5

Brak wartości


,sensor_id,lat,lon,pm25,measured_at
3,PM-006,50.674051,NaN,25.7,2026-10-01 08:03:00
54,PM-095,NaN,17.935210,25.8,2026-10-01 08:18:00
96,PM-006,50.674051,NaN,25.7,2026-10-01 08:03:00
97,PM-123,50.650780,NaN,17.9,2026-10-01 08:18:00
123,PM-039,NaN,17.935817,17.7,2026-10-01 08:05:00
137,PM-096,NaN,17.897319,16.1,2026-10-01 08:52:00
138,PM-008,50.695501,NaN,19.9,2026-10-01 08:01:00


## CZYSZCZENIE DANYCH

In [7]:
# usuniecie duplikatów
df = df.drop_duplicates().copy()

#Konwersja danych na typ liczbowy
for col in ["lat", "lon", "pm25"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# usuwanie rekordów z brakującymi wartosciami
df = df.dropna(subset=["lat", "lon", "pm25"])

# zapis logiczny dla wartości z przedziałów
swapped = df["lat"].between(14, 24.5) & df["lon"].between(49, 55)
print("Przed zmianą:")
display(df.loc[swapped, ["lat", "lon"]])

# zmiana wspolrzednych
df.loc[swapped, ["lat", "lon"]] = df.loc[swapped, ["lon", "lat"]].to_numpy()

# Po zmianie
print("Po zmianie:")
display(df.loc[swapped, ["lat", "lon"]])

# Usuwanie niepoprawnych danych
df = df[df["pm25"] >= 0]
df = df[df["lat"].between(49, 55) & df["lon"].between(14, 24.5)]

# Konwersja na GeoDataFrame
sensors = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

print("Liczba rekordów po czyszczeniu:", len(sensors))

Przed zmianą:


,lat,lon
5,17.873510,50.677587
61,17.947814,50.653409
102,17.923944,50.673200


Po zmianie:


,lat,lon
5,50.677587,17.873510
61,50.653409,17.947814
102,50.673200,17.923944


Liczba rekordów po czyszczeniu: 130


## MAPA + TABELA

In [8]:
# Base map - OpenStreetMap
m = folium.Map(
    location=[50.675, 17.922],
    zoom_start=12,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Mapa satelitarna",
    overlay=False,
    control=True
).add_to(m)

# District boundaries
districts.to_crs(4326).explore(
    m=m,
    style_kwds={"fillOpacity": 0.1},
    name="Dzielnice"
)

# PM2.5 sensors
sensors.explore(
    m=m,
    column="pm25",
    cmap="Oranges",
    tooltip=True,
    popup=True,
    legend=True,
    name="Czujniki PM2.5"
)

# Schools
schools.to_crs(4326).explore(
    m=m,
    color="blue",
    marker_type="marker",
    tooltip=True,
    popup=True,
    name="Szkoły"
)

# Layer switcher
folium.LayerControl().add_to(m)

# Display map
display(m)

# Display sensor table
display(sensors.drop(columns="geometry"))

,sensor_id,lat,lon,pm25,measured_at
0,PM-042,50.675603,17.906425,23.5,2026-10-01 08:36:00
1,PM-106,50.632283,17.919674,12.4,2026-10-01 08:22:00
2,PM-130,50.635608,18.020378,19.4,2026-10-01 08:24:00
4,PM-025,50.676438,17.938766,16.8,2026-10-01 08:13:00
5,PM-137,50.677587,17.873510,14.4,2026-10-01 08:06:00
...,...,...,...,...,...
140,PM-076,50.650969,17.955770,18.8,2026-10-01 08:30:00
141,PM-044,50.665153,17.959465,17.9,2026-10-01 08:12:00
142,PM-023,50.683927,17.916317,18.1,2026-10-01 08:38:00
143,PM-073,50.657254,17.934440,25.1,2026-10-01 08:46:00
